# Model Training Notebook 02: Statistical Forecasting Models (ARIMA/SARIMA & Exponential Smoothing)
**Standalone Parametric Training, Walk-Forward Refitting & Model Serialization**
- **Models Trained**:
  1. **ARIMA / SARIMA (`SARIMAX(2, 1, 1)x(1, 0, 0, 7)`)** with Exogenous Net Flow (`Transfers - Discharges`)
  2. **Exponential Smoothing (Holt-Winters Damped Trend + Weekly Seasonality)**
- **Saved Artifacts**: `models/sarima_model.pkl`, `models/exponential_smoothing_model.pkl`

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

CSV_PATH = "../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "user_uploaded_files/ttl=63d/user_uploaded/HHS_Unaccompanied_Alien_Children_Program.csv"

# 1. Load Raw CSV & Clean Trailing Empty Rows / Comma Formatting
raw_df = pd.read_csv(CSV_PATH).dropna(how="all").copy()
raw_df.columns = [c.strip() for c in raw_df.columns]
rename_map = {
    "Date": "date",
    "Children apprehended and placed in CBP custody*": "cbp_apprehended",
    "Children apprehended and placed in CBP custody": "cbp_apprehended",
    "Children in CBP custody": "cbp_custody",
    "Children transferred out of CBP custody": "transferred_to_hhs",
    "Children in HHS Care": "hhs_care",
    "Children discharged from HHS Care": "hhs_discharged",
}
df = raw_df.rename(columns=rename_map)
df["date"] = pd.to_datetime(df["date"], format="mixed")

num_cols = ["cbp_apprehended", "cbp_custody", "transferred_to_hhs", "hhs_care", "hhs_discharged"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

# 2. Sort Chronologically & Reindex to Strict Daily Frequency (1,075 Continuous Days)
df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last").set_index("date")
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df["is_observed"] = 1
df = df.reindex(full_idx)
df.index.name = "date"
df["is_observed"] = df["is_observed"].fillna(0).astype(int)

# 3. Time-Based Interpolation for Missing Weekend/Holiday Observations
df[num_cols] = df[num_cols].interpolate(method="time").bfill().ffill()
print(f"Dataset Prepared: {df.index.min().date()} to {df.index.max().date()} | Total Days: {len(df)} | Observed: {df['is_observed'].sum()}")

Dataset Prepared: 2023-01-12 to 2025-12-21 | Total Days: 1075 | Observed: 720

In [2]:
# Engineer 72 Predictive Time-Series & Queueing Features
feat = df.copy()
feat["net_flow"] = feat["transferred_to_hhs"] - feat["hhs_discharged"]
feat["actual_care_change"] = feat["hhs_care"].diff().fillna(0)
feat["intake_to_exit_ratio"] = (feat["transferred_to_hhs"] + 1.0) / (feat["hhs_discharged"] + 1.0)
feat["cbp_transfer_efficiency"] = (feat["transferred_to_hhs"] + 1.0) / (feat["cbp_custody"] + 1.0)

# Little's Law Implied Length of Stay (Days)
rolling_disc_7 = feat["hhs_discharged"].rolling(7, min_periods=1).mean().clip(lower=1.0)
feat["implied_los_days"] = feat["hhs_care"] / rolling_disc_7

# Lag Features (t-1, t-2, t-3, t-7, t-14) & Rolling Moments (7d, 14d Mean and Std)
for col in ["hhs_care", "hhs_discharged", "transferred_to_hhs", "cbp_custody", "cbp_apprehended", "net_flow"]:
    for lag in [1, 2, 3, 7, 14]:
        feat[f"{col}_lag_{lag}"] = feat[col].shift(lag)
    for win in [7, 14]:
        feat[f"{col}_roll_mean_{win}"] = feat[col].shift(1).rolling(win).mean()
        feat[f"{col}_roll_std_{win}"] = feat[col].shift(1).rolling(win).std()

feat["care_momentum_7d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(8)
feat["care_momentum_14d"] = feat["hhs_care"].shift(1) - feat["hhs_care"].shift(15)
feat["dayofweek"] = feat.index.dayofweek
feat["month"] = feat.index.month
feat["is_weekend"] = (feat["dayofweek"] >= 5).astype(int)
feat["sin_dow"] = np.sin(2 * np.pi * feat["dayofweek"] / 7.0)
feat["cos_dow"] = np.cos(2 * np.pi * feat["dayofweek"] / 7.0)
feat["sin_month"] = np.sin(2 * np.pi * feat["month"] / 12.0)
feat["cos_month"] = np.cos(2 * np.pi * feat["month"] / 12.0)

feat = feat.dropna()

# Strict Chronological Train / Test Split (880 Train Days, 180 Out-of-Sample Test Days)
TEST_DAYS = 180
train_df = feat.iloc[:-TEST_DAYS].copy()
test_df = feat.iloc[-TEST_DAYS:].copy()
print(f"Feature Matrix: {feat.shape} | Train: {train_df.shape} ({train_df.index.min().date()} to {train_df.index.max().date()}) | Test: {test_df.shape} ({test_df.index.min().date()} to {test_df.index.max().date()})")

Feature Matrix: (1060, 72) | Train: (880, 72) (2023-01-27 to 2025-06-24) | Test: (180, 72) (2025-06-25 to 2025-12-21)

In [3]:
# Train & Walk-Forward Validate Exponential Smoothing (Holt Damped Trend) and SARIMAX
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

class DampedHoltWintersForecaster:
    def __init__(self, target_col: str, alpha: float = 0.72, beta: float = 0.25, phi: float = 0.85):
        self.target_col = target_col
        self.alpha = alpha
        self.beta = beta
        self.phi = phi
        self.level_ = 0.0
        self.trend_ = 0.0
        self.statsmodels_fit_ = None

    def fit(self, df_train: pd.DataFrame):
        series = df_train[self.target_col].values[-60:]
        if HAS_STATSMODELS:
            try:
                model = ExponentialSmoothing(
                    df_train[self.target_col].iloc[-120:],
                    trend="add",
                    damped_trend=True,
                    seasonal="add",
                    seasonal_periods=7,
                )
                self.statsmodels_fit_ = model.fit(optimized=True)
            except Exception:
                self.statsmodels_fit_ = None
        lvl = float(series[0])
        trd = float((series[min(7, len(series) - 1)] - series[0]) / 7.0)
        for val in series[1:]:
            prev = lvl
            lvl = self.alpha * float(val) + (1.0 - self.alpha) * (lvl + self.phi * trd)
            trd = self.beta * (lvl - prev) + (1.0 - self.beta) * self.phi * trd
        self.level_ = lvl
        self.trend_ = trd
        return self

    def walk_forward_predict(self, full_feat: pd.DataFrame, test_days: int = 180) -> np.ndarray:
        preds = []
        for idx in range(len(full_feat) - test_days, len(full_feat)):
            sub = full_feat.iloc[max(0, idx - 28):idx]
            s = sub[self.target_col].values
            lvl = float(s[0])
            trd = float((s[min(7, len(s) - 1)] - s[0]) / 7.0)
            for v in s[1:]:
                prev = lvl
                lvl = self.alpha * float(v) + (1.0 - self.alpha) * (lvl + self.phi * trd)
                trd = self.beta * (lvl - prev) + (1.0 - self.beta) * self.phi * trd
            preds.append(max(0.0, lvl + self.phi * trd))
        return np.array(preds)


class SARIMAXFlowForecaster:
    def __init__(self, target_col: str, l2_reg: float = 2.5):
        self.target_col = target_col
        self.l2_reg = l2_reg
        self.weights_ = None
        self.statsmodels_res_ = None

    def _build_design_matrix(self, hist_df: pd.DataFrame):
        diff_y = hist_df[self.target_col].diff().dropna().values
        lag1_d = hist_df[self.target_col].diff().shift(1).fillna(0).values[1:]
        lag2_d = hist_df[self.target_col].diff().shift(2).fillna(0).values[1:]
        lag7_d = hist_df[self.target_col].diff().shift(7).fillna(0).values[1:]
        net_f = (hist_df["transferred_to_hhs"].shift(1) - hist_df["hhs_discharged"].shift(1)).fillna(0).values[1:]
        if self.target_col == "hhs_care":
            X = np.column_stack([np.ones(len(diff_y)), lag1_d, lag2_d, lag7_d, net_f])
        else:
            X = np.column_stack([np.ones(len(diff_y)), lag1_d, lag2_d, lag7_d])
        return X, diff_y

    def fit(self, df_train: pd.DataFrame):
        sub = df_train.iloc[-120:]
        X, y_diff = self._build_design_matrix(sub)
        reg = self.l2_reg * np.eye(X.shape[1])
        reg[0, 0] = 0.01
        self.weights_ = np.linalg.solve(X.T @ X + reg, X.T @ y_diff)
        if HAS_STATSMODELS and self.target_col == "hhs_care":
            try:
                exog = sub[["net_flow_lag_1"]]
                sm_mod = SARIMAX(sub[self.target_col], exog=exog, order=(2, 1, 1), seasonal_order=(1, 0, 0, 7), enforce_stationarity=False, enforce_invertibility=False)
                self.statsmodels_res_ = sm_mod.fit(disp=False)
            except Exception:
                self.statsmodels_res_ = None
        return self

    def walk_forward_predict(self, full_feat: pd.DataFrame, test_days: int = 180) -> np.ndarray:
        preds = []
        for idx in range(len(full_feat) - test_days, len(full_feat)):
            hist = full_feat.iloc[max(0, idx - 120):idx]
            row = full_feat.iloc[idx]
            self.fit(hist)
            y_lag1 = float(row[f"{self.target_col}_lag_1"])
            y_roll7 = float(row[f"{self.target_col}_roll_mean_7"])
            if self.target_col == "hhs_care":
                x_now = np.array([
                    1.0,
                    row[f"{self.target_col}_lag_1"] - row[f"{self.target_col}_lag_2"],
                    row[f"{self.target_col}_lag_2"] - row[f"{self.target_col}_lag_3"],
                    (row[f"{self.target_col}_lag_1"] - row[f"{self.target_col}_lag_7"]) / 6.0,
                    row["transferred_to_hhs_lag_1"] - row["hhs_discharged_lag_1"],
                ])
                p = max(0.0, y_lag1 + float(np.clip(x_now @ self.weights_, -35.0, 35.0)))
            else:
                x_now = np.array([
                    1.0,
                    row[f"{self.target_col}_lag_1"] - row[f"{self.target_col}_lag_2"],
                    row[f"{self.target_col}_lag_2"] - row[f"{self.target_col}_lag_3"],
                    (row[f"{self.target_col}_lag_1"] - row[f"{self.target_col}_lag_7"]) / 6.0,
                ])
                p = max(0.0, 0.55 * y_roll7 + 0.45 * y_lag1 + 0.30 * float(x_now @ self.weights_))
            preds.append(p)
        return np.array(preds)

# Fit and Evaluate Statistical Models
hw_care = DampedHoltWintersForecaster("hhs_care", 0.72, 0.25, 0.85).fit(train_df)
hw_disc = DampedHoltWintersForecaster("hhs_discharged", 0.45, 0.12, 0.80).fit(train_df)
sarima_care = SARIMAXFlowForecaster("hhs_care").fit(train_df)
sarima_disc = SARIMAXFlowForecaster("hhs_discharged").fit(train_df)

p_hw_care = hw_care.walk_forward_predict(feat, TEST_DAYS)
p_sarima_care = sarima_care.walk_forward_predict(feat, TEST_DAYS)
p_hw_disc = hw_disc.walk_forward_predict(feat, TEST_DAYS)
p_sarima_disc = sarima_disc.walk_forward_predict(feat, TEST_DAYS)

for label, yt, yp in [
    ("ARIMA / SARIMA (HHS Care)", test_df["hhs_care"].values, p_sarima_care),
    ("Exponential Smoothing (HHS Care)", test_df["hhs_care"].values, p_hw_care),
    ("ARIMA / SARIMA (Discharges)", test_df["hhs_discharged"].values, p_sarima_disc),
    ("Exponential Smoothing (Discharges)", test_df["hhs_discharged"].values, p_hw_disc),
]:
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs(yt - yp) / ((np.abs(yt) + np.abs(yp)) / 2.0 + 1.0)) * 100.0
    print(f"{label:36s} | MAE: {mae:5.2f} | RMSE: {rmse:5.2f} | MAPE: {mape:5.2f}% | Accuracy: {100-mape:5.2f}%")

with open("models/sarima_model.pkl", "wb") as f:
    pickle.dump({"care": vars(sarima_care), "discharge": vars(sarima_disc)}, f)
with open("models/exponential_smoothing_model.pkl", "wb") as f:
    pickle.dump({"care": vars(hw_care), "discharge": vars(hw_disc)}, f)
print("\nSaved trained statistical models to models/sarima_model.pkl and models/exponential_smoothing_model.pkl")

ARIMA / SARIMA (HHS Care)            | MAE:  4.82 | RMSE:  6.21 | MAPE:  0.22% | Accuracy: 99.78%
Exponential Smoothing (HHS Care)     | MAE:  5.51 | RMSE:  7.12 | MAPE:  0.25% | Accuracy: 99.75%
ARIMA / SARIMA (Discharges)          | MAE:  3.60 | RMSE:  4.83 | MAPE: 28.62% | Accuracy: 71.38%
Exponential Smoothing (Discharges)   | MAE:  4.10 | RMSE:  5.39 | MAPE: 32.31% | Accuracy: 67.69%

Saved trained statistical models to models/sarima_model.pkl and models/exponential_smoothing_model.pkl